# Phase 1: Shared Ablation (C/D/F on EFT)

4 tasks × 3 modality combos × 2 splits × 3 seeds = **72 runs**

| Task | Improvement |
|------|-------------|
| `eft_F` | Task-Aware Head (trend分支加时间差分) |
| `eft_D` | Multi-Scale Temporal Encoder (多尺度膨胀卷积) |
| `eft_C` | Contrastive Alignment (跨模态InfoNCE辅助loss) |
| `eft_CDF` | C + D + F 全部叠加 |

Modalities: `[video]`, `[video, telem]`, `[video, km, telem]`

Baseline EFT results already in `runs/baselines/`.

Disconnect-safe: re-run Cell 3 to resume from where it left off.

In [2]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
/content/drive/MyDrive/ProjectExperiment
NVIDIA A100-SXM4-80GB, 81920 MiB


In [3]:
# Cell 2: Dry run — preview plan, verify no import errors
!python scripts/run_experiment.py \
    --sweep configs/sweeps/phase1_ablation.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation \
    --dry_run 2>&1 | tail -10

!echo "---"
!python scripts/run_experiment.py \
    --sweep configs/sweeps/phase1_ablation.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation \
    --dry_run 2>&1 | grep -c '\[RUN\]'

  [RUN] eft_C / triple_video_km_telem / seed2
  [RUN] eft_CDF / single_video / seed0
  [RUN] eft_CDF / single_video / seed1
  [RUN] eft_CDF / single_video / seed2
  [RUN] eft_CDF / dual_video_telem / seed0
  [RUN] eft_CDF / dual_video_telem / seed1
  [RUN] eft_CDF / dual_video_telem / seed2
  [RUN] eft_CDF / triple_video_km_telem / seed0
  [RUN] eft_CDF / triple_video_km_telem / seed1
  [RUN] eft_CDF / triple_video_km_telem / seed2
---
72


In [4]:
# Cell 3: Run experiments (re-run this cell to resume after disconnect)
!python scripts/run_experiment.py \
    --sweep configs/sweeps/phase1_ablation.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation

Sweep plan: 72 runs total, 0 already completed, 72 to run


[1/72] [cross_subject] eft_F / single_video / seed0
torch.compile enabled
Model parameters: 9,463,814 (trainable: 9,463,814)
Train samples: 605, Val samples: 204
Run directory: /content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation/cross_subject/eft_F_3seed/single_video/2026-04-03_22-34-15__amucs_seq_multitask__eft__video__seed0
/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
Epoch 1/40 | train_balanced_acc_mean: 0.3320 | train_balanced_acc_state: 0.3332 | train_balanced_acc_trend: 0.3309 | train_f1_mean: 0.2727 | train_f1_state: 0.2587 | train_f1_trend: 0.2868 | train_loss: 2.2649 | train_macro_f1_mean: 0.2727 | train_macro_f1_state: 0.2587 | train_macro_f1_trend: 0.2868 | val_balanced_acc_me

In [5]:
# Cell 4: Check progress (run anytime)
import glob
from collections import Counter

RUNS_ROOT = '/content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation'
TOTAL = 72  # 4 tasks x 3 modality combos x 2 splits x 3 seeds

done = glob.glob(f'{RUNS_ROOT}/**/metrics.json', recursive=True)
print(f'Completed: {len(done)} / {TOTAL}\n')

# Per split + task breakdown
tasks = Counter()
for p in done:
    parts = p.replace('\\', '/').split('/')
    split = 'unknown'
    task = 'unknown'
    for part in parts:
        if part in ('cross_subject', 'within_subject'):
            split = part
        if '_3seed' in part:
            task = part.replace('_3seed', '')
    tasks[f'{split}/{task}'] += 1

expected_per_task = 3 * 3  # 3 modality combos x 3 seeds
for key, count in sorted(tasks.items()):
    print(f'  {key}: {count}/{expected_per_task}')

Completed: 70 / 72

  cross_subject/eft_C: 9/9
  cross_subject/eft_CDF: 8/9
  cross_subject/eft_D: 9/9
  cross_subject/eft_F: 9/9
  within_subject/eft_C: 9/9
  within_subject/eft_CDF: 8/9
  within_subject/eft_D: 9/9
  within_subject/eft_F: 9/9


In [6]:
# Cell 5: Quick results comparison vs EFT baseline
import json, glob, numpy as np
from pathlib import Path

PHASE1 = Path('/content/drive/MyDrive/AmuCS_experiment/runs/eft_CDF_ablation')
BASELINE = Path('/content/drive/MyDrive/AmuCS_experiment/runs/baselines')

def collect_results(root, task_prefix, split, modality_dir):
    """Collect 3-seed metrics for a (task, split, modality) combo."""
    pattern = f'{root}/{split}/{task_prefix}_3seed/{modality_dir}/**/metrics.json'
    files = sorted(glob.glob(pattern, recursive=True))
    metrics = []
    for f in files:
        with open(f) as fp:
            metrics.append(json.load(fp))
    return metrics

def summarize(metrics_list, key):
    vals = [m[key] for m in metrics_list if key in m]
    if not vals:
        return '  N/A  '
    return f'{np.mean(vals):.4f}'

# Compare across splits and modality combos
mod_dirs = {
    'V':     'single_video',
    'V+T':   'dual_video_telem',
    'V+K+T': 'triple_video_km_telem',
}

tasks = {
    'EFT (baseline)': ('eft_state_trend', BASELINE),
    'EFT+F':          ('eft_F',           PHASE1),
    'EFT+D':          ('eft_D',           PHASE1),
    'EFT+C':          ('eft_C',           PHASE1),
    'EFT+CDF':        ('eft_CDF',         PHASE1),
}

for split in ['cross_subject', 'within_subject']:
    print(f'\n{"="*70}')
    print(f'  {split}')
    print(f'{"="*70}')
    header = f'{"Task":<18}'
    for mod_label in mod_dirs:
        header += f'  {mod_label:>10} BAcc  {mod_label:>10} F1'
    print(header)
    print('-' * len(header))

    for task_label, (task_prefix, root) in tasks.items():
        row = f'{task_label:<18}'
        for mod_label, mod_dir in mod_dirs.items():
            ms = collect_results(root, task_prefix, split, mod_dir)
            row += f'  {summarize(ms, "test_balanced_acc_mean"):>14}  {summarize(ms, "test_f1_mean"):>10}'
        print(row)


  cross_subject
Task                         V BAcc           V F1         V+T BAcc         V+T F1       V+K+T BAcc       V+K+T F1
------------------------------------------------------------------------------------------------------------------
EFT (baseline)              0.4445      0.4215          0.4472      0.4207          0.4526      0.4300
EFT+F                       0.4406      0.4172          0.4559      0.4421          0.4507      0.4344
EFT+D                       0.4554      0.4393          0.4605      0.4416          0.4504      0.4385
EFT+C                       0.4445      0.4215          0.4525      0.4283          0.4516      0.4164
EFT+CDF                     0.4538      0.4359          0.4609      0.4408          0.4578      0.4321

  within_subject
Task                         V BAcc           V F1         V+T BAcc         V+T F1       V+K+T BAcc       V+K+T F1
---------------------------------------------------------------------------------------------------------